# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


Lane: Search Intent (Lane 4)

Task type: Ranking / scoring with a classification backbone.

Why ranking/scoring, not pure classification?
The framing skill asks: "What decision does this improve?" My decision is "Which pages should a content editor review first for intent-content alignment?" That is a "which ones first?" question, and the skill's table maps that directly to ranking/scoring with a priority score as the target and precision@K as the metric.

Why not clustering or pure classification?
Clustering would find groups of pages but would not tell me which to fix first. No action follows from a cluster alone. Pure classification gives a yes/no per page, but the content team cannot review 30,000 pages. They need a ranked queue. So I use a classifier to produce a probability, then rank by that probability. The probability is the priority score.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Pages: {len(df):,}")
print(f"Declining (observed outcome): {df['is_declining_label'].mean():.1%}")
print()
print("Task type: RANKING/SCORING")
print("Input:  page features (intent, content_type, visibility, CTR, age)")
print("Output: priority score (0-1), sorted descending")
print("Metric: Precision@K on the ranked list")

Pages: 30,000
Declining (observed outcome): 54.2%

Task type: RANKING/SCORING
Input:  page features (intent, content_type, visibility, CTR, age)
Output: priority score (0-1), sorted descending
Metric: Precision@K on the ranked list


## 2. Target or proxy


Target: 'is_declining_label' is binary, 1 = declining, 0 = not.

Where it comes from: 'trend_direction' is derived from 'trend_pct', which is a measured percentage change in real traffic over the trailing 90 days. So this is an observed outcome, not a defined rule. My label is "decline" a proxy for "worth reviewing for intent-alignment," not a direct intent-mismatch label.

Data gotcha observed: 'trend_pct' is NaN for the 'flat' and 'new' directions. The 'up' direction has a max of 44,900 (an extreme outlier), consistent with the skill's warning that rate columns can be misleading. I will NOT use 'trend_pct' as a feature (leakage), so this does not affect my model but it is worth noting.

In [5]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Source: trend_direction (observed from trend_pct)")
print()
print("trend_direction distribution:")
print(df["trend_direction"].value_counts().to_string())
print()
print("trend_pct range per direction (proof the label is observed, not defined):")
print(df.groupby("trend_direction")["trend_pct"].agg(["min", "max", "mean"]).round(2).to_string())
print()
print("Target distribution:")
print(df["is_declining_label"].value_counts().to_string())
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

Source: trend_direction (observed from trend_pct)

trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

trend_pct range per direction (proof the label is observed, not defined):
                   min      max    mean
trend_direction                        
down            -100.0    -20.0  -58.11
flat               NaN      NaN     NaN
new                NaN      NaN     NaN
stable           -20.0     20.0   -3.19
up                20.0  44900.0  190.67

Target distribution:
is_declining_label
1    16262
0    13738
Declining rate: 54.2%


## 3. Success metric


Metric: Precision@50.

Precision@50 = of the top 50 pages my ranking flags, what fraction are actually declining?

Why this metric:
- The action is "review the top pages" but a human has limited time.
- A content team can realistically review ~50 pages per week.
- Precision@K directly measures "if we act on the top K, how many were worth it."
- Accuracy is useless here: ~54% of pages decline, so "everything declines" gets 54% accuracy.

Formula (in words):
- sort pages by my score descending
- take the top 50
- count how many are actually declining
- divide by 50.

What number means "good":
- Hand-rule baseline: 0.68 (from the starter CSV, seen in the baseline skill).
- Acceptable: ≥ 0.75.
- Good: ≥ 0.85.

In [6]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

y = df["is_declining_label"].values

# Baseline from the building baselines skill
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
baseline_score = stale * visible * df["impressions_90d"]

print("Hand-rule baseline — Precision@K:")
for k in (20, 50, 100):
    print(f"  Precision@{k}: {precision_at_k(baseline_score, y, k):.3f}")
print()
print(f"Base rate (declining): {y.mean():.3f}")
print("A model must beat P@50 = 0.680 to be useful. Target: P@50 >= 0.70.")

Hand-rule baseline — Precision@K:
  Precision@20: 0.900
  Precision@50: 0.680
  Precision@100: 0.630

Base rate (declining): 0.542
A model must beat P@50 = 0.680 to be useful. Target: P@50 >= 0.70.


## 4. The unit of analysis, as a real dataframe


One row = one content page (one pseudonymized content item).

Each row carries:
- Content attributes: 'content_type', 'main_intent', 'word_count', 'content_age_days'
- Search performance: 'impressions_90d', 'avg_position', 'ctr'
- Observed outcome: 'trend_direction' (my label source)

The lane's slice: the columns relevant to intent-content analysis.

Observed signal: the dataset has 3 content types and 4 intent types, but only comparison articles pair with informational intent (697 pages). Every other page is a 'keyword article' across the 4 intent types. This means intent-content alignment is measurable, but narrow.

I can compare informational × comparison article vs informational × keyword article directly.

In [8]:
lane_cols = [
    "content_id", "client_id",
    "content_type", "main_intent",
    "impressions_90d", "avg_position", "ctr",
    "word_count", "content_age_days",
    "trend_direction", "is_declining_label",
]

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Unit of analysis: 1 row = 1 content page")
print()

print("MY LANE'S SLICE: first 5 pages")
print(df[lane_cols].head().to_string(index=False))
print()

print("INTENT x CONTENT_TYPE (my core signal)")
print(df.groupby(["main_intent", "content_type"]).size().unstack(fill_value=0).to_string())
print()

print("TARGET COLUMN (what I predict)")
print(df[["content_id", "content_type", "main_intent",
          "trend_direction", "is_declining_label"]].head(8).to_string(index=False))

Shape: 30,000 rows x 45 columns
Unit of analysis: 1 row = 1 content page

MY LANE'S SLICE: first 5 pages
          content_id         client_id    content_type   main_intent  impressions_90d  avg_position  ctr  word_count  content_age_days trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc keyword article transactional             3803          10.6 0.76      3221.0               187            down                   1
content_a1fb4e703a9e client_4e07408562 keyword article informational            15320          20.3 0.05      2481.0               445            down                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article informational            12581          36.5 0.09      3515.0               141            down                   1
content_331d6c4de07b client_19581e27de keyword article    commercial            11751           6.2 0.49         NaN               463          stable                   0
content_d99b7a2d90ca client_3fdba35f04 k

## 5. Why ML beats a fixed rule here


A learned model can:
- Detect rare intent-content combinations (potential anomalies).
- Weigh CTR against position against age instead of AND-ing thresholds.
- Learn which combinations actually correlate with decline.

A fixed rule cannot do any of those, it can only check the boxes a human wrote down.

Honest finding from the starter CSV: a hand rule (stale × visible × impressions) reaches Precision@50 = 0.680, while a depth-2 decision tree reaches 0.600. On this quick test, the hand rule wins. That is a real result, not noise and it is what we are doing now: building a fair baseline, then see if a properly validated model beats it on the same split and the same metric.

In [10]:
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURES = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]

X = df[FEATURES].replace([np.inf, -np.inf], np.nan)
# avg_position = 0 means "no data" treated as missing, not rank 0
X.loc[df["avg_position"] == 0, "avg_position"] = np.nan
X = X.fillna(-1)  # sentinel for "missing"

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

print("Quick in-sample check — Precision@50:")
print(f"  Hand rule : {precision_at_k(baseline_score, y, 50):.3f}")
print(f"  Tree (d=2): {precision_at_k(tree_score, y, 50):.3f}")
print()
print("The tree's readable rule:")
print(export_text(tree, feature_names=FEATURES))

Quick in-sample check — Precision@50:
  Hand rule : 0.680
  Tree (d=2): 0.600

The tree's readable rule:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.